In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

DATA_DIR = os.path.join(r"D:\Upskill\Mini_Projects\intelligent-predictive-maintenance-system\CMAPSS_Data")

In [ ]:
train_data = pd.read_csv(os.path.join(DATA_DIR, "train_FD001.txt"), sep=" ", header=None)
train_data

In [ ]:
train_data = train_data.dropna(axis=1)
train_data

In [ ]:
#Get the column names from the dataset documentation
column_names = ["engine_id", "cycle"] + [f"operational_setting_{i}" for i in range(1, 4)] + [f"sensor_{i}" for i in range(1, 22)]
column_names

In [ ]:
#Assign column names to the DataFrame
train_data.columns = column_names
train_data

In [ ]:
# Plotting the sensor data for 1 engine
# Some sensor values constant -> Useless
engine_1 = train_data[train_data['engine_id'] == 1]
sensor_cols = [f'sensor_{i}' for i in range(1, 22)]

engine_1.plot(
    x = "cycle",
    y = sensor_cols,
    legend = True
)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

In [ ]:
# Finding RUL (Remaining Useful Life) using the given data
max_cycles = train_data.groupby('engine_id')["cycle"].max() #Give max cycles of each engine as a pandas Series
print(max_cycles)
print('---------------------------------------------')

# Adding an RUL column that tells how close an engine is to failure. Creating ground truth for the training
train_data['RUL'] = train_data.apply(lambda row:max_cycles[row['engine_id']] - row['cycle'], axis = 1)
print(train_data[['engine_id','cycle', 'RUL']])
print('---------------------------------------------')

# Adding a boolean column to tell if the engine is close to failure or not based on a threshold
threshold = 30
train_data['failure_risk'] = (train_data['RUL'] <= 30).astype(int) # Convert to int as we need numeric data to train model
print(train_data.loc[train_data['engine_id'] == 1, ['engine_id','cycle', 'RUL', 'failure_risk']]) #.loc[] takes input as [row, column]. filter by row first then by column

In [ ]:
# Adding RUL for test dataset in the same way

test_data = pd.read_csv(os.path.join(DATA_DIR, "test_FD001.txt"), sep=" ", header=None)
RUL_data = pd.read_csv(os.path.join(DATA_DIR, "RUL_FD001.txt"), header=None)

test_data.dropna(axis=1, inplace=True)
RUL_data.dropna(axis=1, inplace=True)

test_data.columns = column_names

max_cycles = test_data.groupby('engine_id')['cycle'].max()
max_cycles = max_cycles + RUL_data[0].values

test_data['RUL'] = test_data.apply(lambda row:max_cycles[row['engine_id']] - row['cycle'], axis = 1)

threshold = 30
test_data['failure_risk'] = (test_data['RUL'] <= 30).astype(int) # Convert to int as we need numeric data to train model

In [ ]:
print(train_data.shape)
print(test_data.shape)

In [ ]:
print(train_data['engine_id'].nunique())
print(test_data['engine_id'].nunique())